# Is the constraint the corrupted fraction or the client count?

Section 5.9 argues the advantage tracks the fraction of corrupted data,
not how many hospitals there are. The evidence is two measurements that
agree at 25% -- 1.46x from the K sweep and 1.49x from the multi-site
sweep -- but both sit at K = 4, so a reviewer can fairly say the claim
rests on one federation size.

This holds the fraction at 25% and moves the client count: 2 of 8,
3 of 12, 4 of 16. If the ratio stays near 1.5x the fraction reading is
right. If it decays with K, it is not, and Section 5.9 has to be
withdrawn.

Total data is held at 4000 samples throughout, matching the existing
scaling sweep (spc 1000 / 500 / 333 / 250) so the only thing changing
is how the same data is split and how much of it is corrupted.


In [ ]:
import glob, os, shutil, sys
REPO='/kaggle/working/repo'
hits=glob.glob('/kaggle/input/**/paper/build_paper.py', recursive=True)
assert hits, 'attach the aha-fed-code dataset'
src=os.path.dirname(os.path.dirname(hits[0]))
if not os.path.isdir(os.path.join(REPO,'paper')):
    shutil.copytree(src, REPO, dirs_exist_ok=True)
os.chdir(REPO); sys.path.insert(0, REPO)
print('staged:', os.getcwd())


In [ ]:
!pip install -q medmnist opacus

In [ ]:
import subprocess, sys
PROBE=('import torch;'
       'a=torch.randn(4,3,8,8,device="cuda");'
       'w=torch.randn(2,3,3,3,device="cuda");'
       'torch.nn.functional.conv2d(a,w).sum().item();'
       'print(torch.cuda.get_device_name(0), torch.__version__)')
r=subprocess.run([sys.executable,'-c',PROBE],capture_output=True,text=True)
print(r.stdout.strip() or r.stderr.strip()[-400:])
if r.returncode!=0:
    subprocess.run([sys.executable,'-m','pip','install','-q',
                    'torch==2.4.1','torchvision==0.19.1',
                    '--index-url','https://download.pytorch.org/whl/cu118'],check=True)
    r=subprocess.run([sys.executable,'-c',PROBE],capture_output=True,text=True)
    print(r.stdout.strip() or r.stderr.strip()[-400:])
    if r.returncode!=0: raise SystemExit('no working torch/GPU combination')


In [ ]:
from paper.run_real_experiments import run_one, OUT_DIR

SEEDS=[42,123,456,7,13,21,99,2024,31337,555]

# 25% corrupted at three federation sizes beyond the K=4 already run.
# samples_per_client keeps the
# total at ~4000 so this varies the split, not the amount of data --
# the existing scaling sweep uses the same convention.
GRID=[(8,2,500),(12,3,333),(16,4,250)]

for K,nn,spc in GRID:
    for s in SEEDS:
        common=dict(dataset='pathmnist', num_classes=9,
                    num_clients=K, num_noisy=nn, noisy_client=K-1,
                    samples_per_client=spc, seed=s)
        run_one(OUT_DIR/f'frac_fedavg_k{K}_n{nn}_s{s}.json',
                aggregator='fedavg', **common)
        run_one(OUT_DIR/f'frac_aha_k{K}_n{nn}_s{s}.json',
                aggregator='adaptive', alpha=1.0, beta=4.0, gamma=4.0,
                **common)
    print(f'K={K}, {nn} corrupted ({nn/K:.0%}) done', flush=True)


In [ ]:
import glob, json, re, statistics as st
from pathlib import Path
from scipy.stats import wilcoxon

def by_seed(pat, key='ece'):
    out={}
    for f in sorted(glob.glob(pat)):
        d=json.loads(Path(f).read_text())
        fin=d.get('final') or {}
        m=re.search(r'_s(\d+)$', Path(f).stem)
        if fin.get(key) is not None:
            out[m.group(1) if m else '42']=fin[key]
    return out

print('The 25% corrupted fraction across federation sizes')
print('  K   corrupted   AHA-Fed   FedAvg    ratio    p        n')
rows=[]
for K,nn in [(8,2),(12,3),(16,4)]:
    a=by_seed(f'runs/paper/real/frac_aha_k{K}_n{nn}_s*.json')
    v=by_seed(f'runs/paper/real/frac_fedavg_k{K}_n{nn}_s*.json')
    c=sorted(set(a)&set(v))
    if not c: continue
    x=[a[k] for k in c]; y=[v[k] for k in c]
    p=wilcoxon(x,y).pvalue if len(c)>1 else float('nan')
    r=st.mean(y)/st.mean(x)
    rows.append((K,nn,r,p,len(c)))
    print(f'  {K:2d}   {nn} ({nn/K:.0%})   {st.mean(x):.4f}   {st.mean(y):.4f}   '
          f'{r:.2f}x   p={p:.4f}  n={len(c)}')

print()
print('existing 25% points, both at K=4: 1.46x (scaling), 1.49x (multi-site)')
if rows:
    lo=min(r for _,_,r,_,_ in rows)
    hi=max(r for _,_,r,_,_ in rows)
    print(f'new points span {lo:.2f}x to {hi:.2f}x')
    print('fraction reading holds' if lo>1.25 else
          'RATIO DECAYS WITH K -- Section 5.9 needs revising')


In [ ]:
!cd /kaggle/working/repo && zip -qr /kaggle/working/frac_results.zip runs/paper/real
!ls -lh /kaggle/working/frac_results.zip
